### Import

In [182]:
import os; import pandas as pd
pd.options.display.float_format = '{:.3f}'.format
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)
import numpy as np; import matplotlib.pyplot as plt
import gurobipy as gp; from gurobipy import GRB
from itertools import product; from tqdm import tqdm
import importlib
import functions_utils; import functions_data
import functions_optimize; import functions_eval
importlib.reload(functions_data); importlib.reload(functions_optimize)
importlib.reload(functions_eval); importlib.reload(functions_utils)
from functions_utils import *; from functions_data import *
from functions_optimize import *; from functions_eval import *
import time

S = 15
LEVEL = "high"
SEED = 1

generation_data, I, T = load_generation_data(date_filter="2022-07-18")
R, P_RT, K, K0, M1, M2 = load_parameters(I, T, generation_data, S, LEVEL, SEED)
P_DA, P_PN = load_price_data(P_RT)

print("-"*100)
print("[Individual Participation Model optimization]")
x_ind, yp_ind, ym_ind, z_ind, zc_ind, zd_ind, OBJ_IND = optimize_individually_forall(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)

✅ 총 5개 파일을 불러왔습니다: 1201.csv, 137.csv, 401.csv, 524.csv, 89.csv
📊 데이터 Shape: I=5, T=24, S=15
✅ 시뮬레이션 초기화 완료: S=15, Randomness='high', Random Seed=1, M1=696.00, M2=1955.00
----------------------------------------------------------------------------------------------------
[Individual Participation Model optimization]


Optimizing individually for each target_i:   0%|          | 0/5 [00:00<?, ?it/s]

Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  20%|██        | 1/5 [00:00<00:00,  8.41it/s]

Optimal solution found for target_i=0! Objective value: 249295.10293220685
Set parameter MIPGap to value 1e-07


...

Optimal solution found for target_i=2! Objective value: 394589.5168807215
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i:  80%|████████  | 4/5 [00:00<00:00,  9.10it/s]

Optimal solution found for target_i=3! Objective value: 496066.30000857136
Set parameter MIPGap to value 1e-07


Optimizing individually for each target_i: 100%|██████████| 5/5 [00:00<00:00,  8.86it/s]

Optimal solution found for target_i=4! Objective value: 193561.4245204222


### Holistic Optimization (쌩)

In [173]:
def optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1):
    set = gp.Model("set_independent")
    set.setParam("MIPGap", 1e-7)

    x_hol = set.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp")
    ym_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
    dp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp")
    dm_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
    
    z_hol = set.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, name="z")
    zc_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, name="zc") ; zd_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, name="zd")

    phi1_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
    phi3_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
    phi5_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
    phi7_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")

    set.update()

    obj_hol = (
        gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) +
        gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
    )
    
    set.setObjective(obj_hol, GRB.MAXIMIZE)

    for i, t, s in product(range(I), range(T), range(S)):
        set.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
        set.addConstr(R[i, t, s] + zd_hol[i, t, s] >= yp_hol[i, t, s] + dp_hol[i, t, s] + zc_hol[i, t, s])
        
        set.addConstr(zd_hol[i, t, s] <= z_hol[i, t, s])
        set.addConstr(zc_hol[i, t, s] <= K[i] - z_hol[i, t, s])
        set.addConstr(z_hol[i, t, s] <= K[i])
        set.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + 0.92 * zc_hol[i, t, s] - zd_hol[i, t, s] / 0.95)
        
        set.addConstr(yp_hol[i, t, s] <= M1 * phi1_hol[i, t, s]) ; set.addConstr(ym_hol[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
        set.addConstr(ym_hol[i, t, s] <= M1 * phi2_hol[i, t, s]) ; set.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
        set.addConstr(zc_hol[i, t, s] <= M1 * phi3_hol[i, t, s]) ; set.addConstr(zd_hol[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
        set.addConstr(dp_hol[i, t, s] <= M1 * phi4_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
        set.addConstr(zc_hol[i, t, s] <= M1 * phi5_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
        set.addConstr(yp_hol[i, t, s] <= M1 * phi6_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))
        set.addConstr(ym_hol[i, t, s] <= M1 * phi7_hol[i, t, s]) ; set.addConstr(dp_hol[i, t, s] <= M1 * (1 - phi7_hol[i, t, s]))

    for i, s in product(range(I), range(S)): set.addConstr(z_hol[i, 0, s] == K0[i])

    balance_constraints = {}
    for t, s in product(range(T), range(S)):
        balance_constraints[t, s] = set.addConstr(gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)),
            name=f"balance_{t}_{s}"
        )

    for i in range(I):
        set.addConstr(
            gp.quicksum(P_DA[t] * x_hol[i, t] for t in range(T)) +
            gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for t in range(T) for s in range(S)) >= OBJ_IND[i]
        )

    set.optimize()
    
    if set.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {set.objVal}")
        
        # 먼저 원래 솔루션 저장
        x_sol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
        yp_sol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        ym_sol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        dp_sol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        dm_sol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        z_sol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
        zc_sol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        zd_sol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        original_objval = set.objVal
        phi1_sol = np.array([[[phi1_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi2_sol = np.array([[[phi2_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi3_sol = np.array([[[phi3_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi4_sol = np.array([[[phi4_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi5_sol = np.array([[[phi5_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi6_sol = np.array([[[phi6_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi7_sol = np.array([[[phi7_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        
        # Dual variable 추출 시도
        try:
            # Method 1: 직접 dual variable 추출 시도
            lambda_dual = {}
            for t, s in product(range(T), range(S)):
                lambda_dual[t, s] = balance_constraints[t, s].Pi
            print("Direct dual extraction successful!")
            
        except AttributeError:
            print("Direct dual extraction failed. Using Model.fixed() method...")
            
            # Method 2: Model.fixed()를 사용하여 이진 변수들을 현재값으로 고정
            fixed_model = set.fixed()
            fixed_model.setParam("OutputFlag", 0)
            
            # 고정된 모델 최적화
            fixed_model.optimize()
            
            # 목적함수값 비교 확인
            if fixed_model.status == GRB.OPTIMAL:
                original_obj = set.objVal
                fixed_obj = fixed_model.objVal
                obj_diff = abs(original_obj - fixed_obj)
                
                print(f"Original MIP objective: {original_obj:.6f}")
                print(f"Fixed LP objective: {fixed_obj:.6f}")
                print(f"Difference: {obj_diff:.10f}")
                
                if obj_diff < 1e-6:
                    print("✅ Objective values match! Fixed model is correct.")
                else:
                    print("⚠️  Warning: Objective values don't match. Check model consistency.")
            
            # Fixed model에서 balance constraint의 dual variable 추출
            lambda_dual = np.zeros((T, S))

            if fixed_model.status == GRB.OPTIMAL:
                # Balance constraint의 이름으로 찾아서 dual variable 추출
                for t, s in product(range(T), range(S)):
                    constr_name = f"balance_{t}_{s}"
                    try:
                        # Constraint를 이름으로 찾기
                        constr = fixed_model.getConstrByName(constr_name)
                        if constr is not None:
                            lambda_dual[t, s] = constr.Pi
                        else:
                            lambda_dual[t, s] = np.nan  # Use np.nan for missing values
                    except:
                        lambda_dual[t, s] = np.nan  # Use np.nan for errors
                        
                print("Model.fixed() dual extraction successful!")
            else:
                print("Fixed model optimization failed. Setting dual to zero.")
                lambda_dual = np.zeros((T, S))

            print("\nInternal Settlement Prices (Dual Variables):")
            # Now you can print from the NumPy array for a cleaner output
            for t in range(T):
                for s in range(S):
                    print(f"λ_{t}(ξ_{s}) = {lambda_dual[t, s]:.4f}")
                
    else:
        print("No optimal solution found.")
        lambda_dual = {(t, s): 0.0 for t in range(T) for s in range(S)}
        x_sol = yp_sol = ym_sol = dp_sol = dm_sol = z_sol = zc_sol = zd_sol = None
        original_objval = None
    
    return (x_sol, yp_sol, ym_sol, dp_sol, dm_sol, z_sol, zc_sol, zd_sol,
            phi1_sol, phi2_sol, phi3_sol, phi4_sol, phi5_sol, phi6_sol, phi7_sol,
            original_objval, lambda_dual)

x_hol, yp_hol, ym_hol, dp_hol, dm_hol, z_hol, zc_hol, zd_hol, phi1_hol, phi2_hol, phi3_hol, phi4_hol, phi5_hol, phi6_hol, phi7_hol, OBJ_HOL, lambda_dual = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)

Set parameter MIPGap to value 1e-07
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-07

Optimize a model with 36440 rows, 25395 columns and 93795 nonzeros
Model fingerprint: 0x9f8e79fa
Variable types: 12795 continuous, 12600 integer (12600 binary)
Coefficient statistics:
  Matrix range     [9e-01, 7e+03]
  Objective range  [3e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+05]
Presolve removed 14228 rows and 8516 columns
Presolve time: 0.21s
Presolved: 22212 rows, 16879 columns, 60537 nonzeros
Variable types: 8666 continuous, 8213 integer (8213 binary)

Root relaxation: objective 1.792233e+06, 10219 iterations, 0.14 seconds (0.20 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  O

In [183]:
def optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1):
    set = gp.Model("set_independent")
    set.setParam("MIPGap", 1e-7)

    x_hol = set.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp")
    ym_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
    dp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp")
    dm_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
    
    z_hol = set.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, name="z")
    zc_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, name="zc") ; zd_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, name="zd")

    phi1_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi1") ; phi2_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
    phi3_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi3") ; phi4_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
    phi5_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi5") ; phi6_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
    phi7_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")

    set.update()

    obj_hol = (
        gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) +
        gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for i in range(I) for t in range(T) for s in range(S))
    )
    
    set.setObjective(obj_hol, GRB.MAXIMIZE)

    for i, t, s in product(range(I), range(T), range(S)):
        set.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
        set.addConstr(R[i, t, s] + zd_hol[i, t, s] >= yp_hol[i, t, s] + dp_hol[i, t, s] + zc_hol[i, t, s])
        
        set.addConstr(zd_hol[i, t, s] <= z_hol[i, t, s])
        set.addConstr(zc_hol[i, t, s] <= K[i] - z_hol[i, t, s])
        set.addConstr(z_hol[i, t, s] <= K[i])
        set.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + 0.92 * zc_hol[i, t, s] - zd_hol[i, t, s] / 0.95)
        
        set.addConstr(yp_hol[i, t, s] <= M1 * phi1_hol[i, t, s]) ; set.addConstr(ym_hol[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
        set.addConstr(ym_hol[i, t, s] <= M1 * phi2_hol[i, t, s]) ; set.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
        set.addConstr(zc_hol[i, t, s] <= M1 * phi3_hol[i, t, s]) ; set.addConstr(zd_hol[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
        set.addConstr(dp_hol[i, t, s] <= M1 * phi4_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
        set.addConstr(zc_hol[i, t, s] <= M1 * phi5_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
        set.addConstr(yp_hol[i, t, s] <= M1 * phi6_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))
        set.addConstr(ym_hol[i, t, s] <= M1 * phi7_hol[i, t, s]) ; set.addConstr(dp_hol[i, t, s] <= M1 * (1 - phi7_hol[i, t, s]))

    for i, s in product(range(I), range(S)): set.addConstr(z_hol[i, 0, s] == K0[i])

    balance_constraints = {}
    for t, s in product(range(T), range(S)):
        balance_constraints[t, s] = set.addConstr(gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)),
            name=f"balance_{t}_{s}"
        )

    for i in range(I):
        set.addConstr(
            gp.quicksum(P_DA[t] * x_hol[i, t] for t in range(T)) +
            gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for t in range(T) for s in range(S)) >= OBJ_IND[i]
        )

    set.optimize()
    
    if set.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {set.objVal}")
        
        # 먼저 원래 솔루션 저장
        x_sol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
        yp_sol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        ym_sol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        dp_sol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        dm_sol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        z_sol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
        zc_sol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        zd_sol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        original_objval = set.objVal
        phi1_sol = np.array([[[phi1_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi2_sol = np.array([[[phi2_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi3_sol = np.array([[[phi3_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi4_sol = np.array([[[phi4_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi5_sol = np.array([[[phi5_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi6_sol = np.array([[[phi6_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        phi7_sol = np.array([[[phi7_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        
        # Dual variable 추출 시도
        try:
            # Method 1: 직접 dual variable 추출 시도
            lambda_dual = {}
            for t, s in product(range(T), range(S)):
                lambda_dual[t, s] = balance_constraints[t, s].Pi
            print("Direct dual extraction successful!")
            
        except AttributeError:
            print("Direct dual extraction failed. Using Model.fixed() method...")
            
            # Method 2: Model.fixed()를 사용하여 이진 변수들을 현재값으로 고정
            fixed_model = set.fixed()
            fixed_model.setParam("OutputFlag", 0)
            
            # 고정된 모델 최적화
            fixed_model.optimize()
            
            # 목적함수값 비교 확인
            if fixed_model.status == GRB.OPTIMAL:
                original_obj = set.objVal
                fixed_obj = fixed_model.objVal
                obj_diff = abs(original_obj - fixed_obj)
                
                print(f"Original MIP objective: {original_obj:.6f}")
                print(f"Fixed LP objective: {fixed_obj:.6f}")
                print(f"Difference: {obj_diff:.10f}")
                
                if obj_diff < 1e-6:
                    print("✅ Objective values match! Fixed model is correct.")
                else:
                    print("⚠️  Warning: Objective values don't match. Check model consistency.")
                
                # 해 비교 추가
                print("\n=== Solution Comparison ===")
                
                # Fixed model에서 해 추출
                fixed_x = {}
                fixed_yp = {}
                fixed_ym = {}
                fixed_dp = {}
                fixed_dm = {}
                fixed_zc = {}
                fixed_zd = {}
                
                for var in fixed_model.getVars():
                    if var.VarName.startswith('x['):
                        indices = var.VarName.replace('x[', '').replace(']', '').split(',')
                        i, t = int(indices[0]), int(indices[1])
                        fixed_x[i, t] = var.X
                    elif var.VarName.startswith('yp['):
                        indices = var.VarName.replace('yp[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_yp[i, t, s] = var.X
                    elif var.VarName.startswith('ym['):
                        indices = var.VarName.replace('ym[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_ym[i, t, s] = var.X
                    elif var.VarName.startswith('dp['):
                        indices = var.VarName.replace('dp[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_dp[i, t, s] = var.X
                    elif var.VarName.startswith('dm['):
                        indices = var.VarName.replace('dm[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_dm[i, t, s] = var.X
                    elif var.VarName.startswith('zc['):
                        indices = var.VarName.replace('zc[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_zc[i, t, s] = var.X
                    elif var.VarName.startswith('zd['):
                        indices = var.VarName.replace('zd[', '').replace(']', '').split(',')
                        i, t, s = int(indices[0]), int(indices[1]), int(indices[2])
                        fixed_zd[i, t, s] = var.X
                
                # 해 차이 계산
                max_diff_x = 0
                max_diff_yp = 0
                max_diff_ym = 0
                max_diff_dp = 0
                max_diff_dm = 0
                max_diff_zc = 0
                max_diff_zd = 0
                
                for i in range(I):
                    for t in range(T):
                        diff = abs(x_sol[i, t] - fixed_x.get((i, t), 0))
                        max_diff_x = max(max_diff_x, diff)
                        
                        for s in range(S):
                            diff = abs(yp_sol[i, t, s] - fixed_yp.get((i, t, s), 0))
                            max_diff_yp = max(max_diff_yp, diff)
                            
                            diff = abs(ym_sol[i, t, s] - fixed_ym.get((i, t, s), 0))
                            max_diff_ym = max(max_diff_ym, diff)
                            
                            diff = abs(dp_sol[i, t, s] - fixed_dp.get((i, t, s), 0))
                            max_diff_dp = max(max_diff_dp, diff)
                            
                            diff = abs(dm_sol[i, t, s] - fixed_dm.get((i, t, s), 0))
                            max_diff_dm = max(max_diff_dm, diff)
                            
                            diff = abs(zc_sol[i, t, s] - fixed_zc.get((i, t, s), 0))
                            max_diff_zc = max(max_diff_zc, diff)
                            
                            diff = abs(zd_sol[i, t, s] - fixed_zd.get((i, t, s), 0))
                            max_diff_zd = max(max_diff_zd, diff)
                
                print(f"Max difference in x: {max_diff_x:.10f}")
                print(f"Max difference in yp: {max_diff_yp:.10f}")
                print(f"Max difference in ym: {max_diff_ym:.10f}")
                print(f"Max difference in dp: {max_diff_dp:.10f}")
                print(f"Max difference in dm: {max_diff_dm:.10f}")
                print(f"Max difference in zc: {max_diff_zc:.10f}")
                print(f"Max difference in zd: {max_diff_zd:.10f}")
                
                total_max_diff = max(max_diff_x, max_diff_yp, max_diff_ym, max_diff_dp, max_diff_dm, max_diff_zc, max_diff_zd)
                if total_max_diff < 1e-6:
                    print("✅ All solutions match! Fixed model solution is correct.")
                else:
                    print("⚠️  Warning: Solutions don't match. Check model consistency.")
            
            # Fixed model에서 balance constraint의 dual variable 추출
            lambda_dual = np.zeros((T, S))

            if fixed_model.status == GRB.OPTIMAL:
                # Balance constraint의 이름으로 찾아서 dual variable 추출
                for t, s in product(range(T), range(S)):
                    constr_name = f"balance_{t}_{s}"
                    try:
                        # Constraint를 이름으로 찾기
                        constr = fixed_model.getConstrByName(constr_name)
                        if constr is not None:
                            lambda_dual[t, s] = constr.Pi
                        else:
                            lambda_dual[t, s] = np.nan  # Use np.nan for missing values
                    except:
                        lambda_dual[t, s] = np.nan  # Use np.nan for errors
                        
                print("Model.fixed() dual extraction successful!")
            else:
                print("Fixed model optimization failed. Setting dual to zero.")
                lambda_dual = np.zeros((T, S))

            print("\nInternal Settlement Prices (Dual Variables):")
            # Now you can print from the NumPy array for a cleaner output
            for t in range(T):
                for s in range(S):
                    print(f"λ_{t}(ξ_{s}) = {lambda_dual[t, s]:.4f}")
                
    else:
        print("No optimal solution found.")
        lambda_dual = {(t, s): 0.0 for t in range(T) for s in range(S)}
        x_sol = yp_sol = ym_sol = dp_sol = dm_sol = z_sol = zc_sol = zd_sol = None
        original_objval = None
    
    return (x_sol, yp_sol, ym_sol, dp_sol, dm_sol, z_sol, zc_sol, zd_sol,
            phi1_sol, phi2_sol, phi3_sol, phi4_sol, phi5_sol, phi6_sol, phi7_sol,
            original_objval, lambda_dual)

x_hol, yp_hol, ym_hol, dp_hol, dm_hol, z_hol, zc_hol, zd_hol, phi1_hol, phi2_hol, phi3_hol, phi4_hol, phi5_hol, phi6_hol, phi7_hol, OBJ_HOL, lambda_dual = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)

Set parameter MIPGap to value 1e-07
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-07

Optimize a model with 36440 rows, 25395 columns and 93795 nonzeros
Model fingerprint: 0x996ffaf6
Variable types: 12795 continuous, 12600 integer (12600 binary)
Coefficient statistics:
  Matrix range     [9e-01, 7e+02]
  Objective range  [3e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 5e+05]
Presolve removed 14228 rows and 8516 columns
Presolve time: 0.22s
Presolved: 22212 rows, 16879 columns, 60537 nonzeros
Variable types: 8666 continuous, 8213 integer (8213 binary)

Root relaxation: objective 1.791417e+06, 10800 iterations, 0.20 seconds (0.28 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  O

In [137]:
case_found = [False] * 7

for i, t, s in product(range(I), range(T), range(S)):
    # 제약조건 1: yp_hol과 ym_hol이 동시에 양수인 경우
    if not case_found[0] and yp_hol[i, t, s] > 0 and ym_hol[i, t, s] > 0:
        print(f"Case 1 발생: i={i}, t={t}, s={s} - yp_hol={yp_hol[i,t,s]}, ym_hol={ym_hol[i,t,s]}")
        case_found[0] = True
    
    # 제약조건 2: ym_hol과 zc_hol이 동시에 양수인 경우
    if not case_found[1] and ym_hol[i, t, s] > 0 and zc_hol[i, t, s] > 0:
        print(f"Case 2 발생: i={i}, t={t}, s={s} - ym_hol={ym_hol[i,t,s]}, zc_hol={zc_hol[i,t,s]}")
        case_found[1] = True
    
    # 제약조건 3: zc_hol과 zd_hol이 동시에 양수인 경우
    if not case_found[2] and zc_hol[i, t, s] > 0 and zd_hol[i, t, s] > 0:
        print(f"Case 3 발생: i={i}, t={t}, s={s} - zc_hol={zc_hol[i,t,s]}, zd_hol={zd_hol[i,t,s]}")
        case_found[2] = True
    
    # 제약조건 4: dp_hol과 dm_hol이 동시에 양수인 경우
    if not case_found[3] and dp_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 4 발생: i={i}, t={t}, s={s} - dp_hol={dp_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}")
        case_found[3] = True
    
    # 제약조건 5: zc_hol과 dm_hol이 동시에 양수인 경우
    if not case_found[4] and zc_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 5 발생: i={i}, t={t}, s={s} - zc_hol={zc_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}")
        case_found[4] = True
    
    # 제약조건 6: yp_hol과 dm_hol이 동시에 양수인 경우
    if not case_found[5] and yp_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 6 발생: i={i}, t={t}, s={s} - yp_hol={yp_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}")
        case_found[5] = True
    
    # 제약조건 7: ym_hol과 dp_hol이 동시에 양수인 경우
    if not case_found[6] and ym_hol[i, t, s] > 0 and dp_hol[i, t, s] > 0:
        print(f"Case 7 발생: i={i}, t={t}, s={s} - ym_hol={ym_hol[i,t,s]}, dp_hol={dp_hol[i,t,s]}")
        case_found[6] = True
    
    # 모든 케이스가 발견되면 조기 종료
    if all(case_found):
        break

# 발견되지 않은 케이스 출력
for i, found in enumerate(case_found, 1):
    if not found:
        print(f"Case {i}: 발생하지 않음")

Case 5 발생: i=1, t=11, s=5 - zc_hol=7.360000006646768e-06, dm_hol=52.880007360000036
Case 1: 발생하지 않음
Case 2: 발생하지 않음
Case 3: 발생하지 않음
Case 4: 발생하지 않음
Case 6: 발생하지 않음
Case 7: 발생하지 않음


In [181]:
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    41.07     0.00     4.23     0.00     0.00     0.00    37.59     0.76    15.09
 9 |   190.27    55.92     0.41     0.00    53.19    53.19   133.94     0.00    48.88
10 |   420.07   232.87    64.30     0.00    57.43    57.43   124.15     1.26   172.10
11 |   598.13   317.00   142.37     0.00    59.66    59.66   153.55    14.78   285.00
12 |   853.33     0.00   768.76     0.00     0.00     0.00    84.58     0.00   410.70
13 |  1300.27     0.00  1599.70     0.00     0.00     0.00     1.21   300.65   488.51
14 |  1405.60   919.00   215.76     0.00   424.53   424.53   276.11     5.27   173.16
15 |   825.67     0.00  1217.29     0.00     0.00     0.00     0.00   391.62   421.64
16 |   759.27   524.00    85.17     0.00   213.47   213.47   159.03     8.93     9.40
17 |   820.33     0.00   955.54     0.00     0.00

### LP로 바꾸려는 중

In [104]:
def optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1):
    set = gp.Model("set")
    set.setParam("MIPGap", 0.001)
    # set.setParam("OutputFlag", 0)

    x_hol = set.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    ep_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="e_plus")
    em_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="e_minus")
    yp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="y_plus")
    ym_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="y_minus")
    z_hol = set.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
    zc_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0,name="z_charge")
    zd_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0,name="z_discharge")
    dp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="d_plus")
    dm_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="d_minus")

    p1_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="p1")
    p2_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="p2")
    p3_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="p3")
    p4_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="p4")

    set.update()

    obj_hol = gp.quicksum(P_DA[t] * gp.quicksum(x_hol[i, t] for i in range(I)) for t in range(T)) + gp.quicksum((1 / S) * (P_RT[t, s] * gp.quicksum(ep_hol[i, t, s] for i in range(I)) - P_PN[t, s] * gp.quicksum(em_hol[i, t, s] for i in range(I))) for t in range(T) for s in range(S))

    set.setObjective(obj_hol, GRB.MAXIMIZE)

    for i, t, s in product(range(I), range(T), range(S)):
        set.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
        set.addConstr(yp_hol[i, t, s] + zc_hol[i, t, s] <= R[i, t, s] + zd_hol[i, t, s])
        set.addConstr(zd_hol[i, t, s] <= z_hol[i, t, s])
        set.addConstr(zc_hol[i, t, s] <= K[i] - z_hol[i, t, s])
        set.addConstr(z_hol[i, t, s] <= K[i])
        set.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + 0.92 * zc_hol[i, t, s] - zd_hol[i, t, s] / 0.95)
        # set.addConstr(yp_hol[i, t, s] <= M1 * p1_hol[i, t, s])
        # set.addConstr(ym_hol[i, t, s] <= M1 * (1 - p1_hol[i, t, s]))
        # set.addConstr(ym_hol[i, t, s] <= M1 * p2_hol[i, t, s])
        # set.addConstr(zc_hol[i, t, s] <= M1 * (1 - p2_hol[i, t, s]))
        # set.addConstr(zc_hol[i, t, s] <= M1 * p3_hol[i, t, s])
        # set.addConstr(zd_hol[i, t, s] <= M1 * (1 - p3_hol[i, t, s]))
        
    for i, s in product(range(I), range(S)):
        set.addConstr(z_hol[i, 0, s] == K0[i])

    for i, t, s in product(range(I), range(T), range(S)):
        set.addConstr(ep_hol[i, t, s] == yp_hol[i, t, s] - dp_hol[i, t, s])
        set.addConstr(em_hol[i, t, s] == ym_hol[i, t, s] - dm_hol[i, t, s])
        # set.addConstr(gp.quicksum(ep_hol[i, t, s] for i in range(I)) <= M2 * p4_hol[i, t, s])
        # set.addConstr(gp.quicksum(em_hol[i, t, s] for i in range(I)) <= M2 * (1 - p4_hol[i, t, s]))
        
    balance_constraints = {}
    for t, s in product(range(T), range(S)):
        balance_constraints[t, s] = set.addConstr(gp.quicksum(dp_hol[i, t, s] for i in range(I)) == gp.quicksum(dm_hol[i, t, s] for i in range(I)),
            name=f"balance_{t}_{s}"
        )

    set.optimize()

    if set.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {set.objVal}")
        
        # 먼저 원래 솔루션 저장
        x_sol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
        ep_sol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        em_sol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        yp_sol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        ym_sol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        dp_sol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        dm_sol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        z_sol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
        zc_sol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        zd_sol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        original_objval = set.objVal
        
        # Dual variable 추출 시도
        try:
            # Method 1: 직접 dual variable 추출 시도
            lambda_dual = {}
            for t, s in product(range(T), range(S)):
                lambda_dual[t, s] = balance_constraints[t, s].Pi
            print("Direct dual extraction successful!")
            
        except AttributeError:
            print("Direct dual extraction failed. Using Model.fixed() method...")
            
            # Method 2: Model.fixed()를 사용하여 이진 변수들을 현재값으로 고정
            fixed_model = set.fixed()
            fixed_model.setParam("OutputFlag", 0)  # 출력 억제
            
            # 고정된 모델 최적화
            fixed_model.optimize()
            
            # 목적함수값 비교 확인
            if fixed_model.status == GRB.OPTIMAL:
                original_obj = set.objVal
                fixed_obj = fixed_model.objVal
                obj_diff = abs(original_obj - fixed_obj)
                
                print(f"Original MIP objective: {original_obj:.6f}")
                print(f"Fixed LP objective: {fixed_obj:.6f}")
                print(f"Difference: {obj_diff:.10f}")
                
                if obj_diff < 1e-6:
                    print("✅ Objective values match! Fixed model is correct.")
                else:
                    print("⚠️  Warning: Objective values don't match. Check model consistency.")
            
            # Fixed model에서 balance constraint의 dual variable 추출
            if fixed_model.status == GRB.OPTIMAL:
                # Balance constraint의 이름으로 찾아서 dual variable 추출
                for t, s in product(range(T), range(S)):
                    constr_name = f"balance_{t}_{s}"
                    try:
                        # Constraint를 이름으로 찾기
                        constr = fixed_model.getConstrByName(constr_name)
                        if constr is not None:
                            lambda_dual[t, s] = constr.Pi
                        else:
                            lambda_dual[t, s] = np.nan  # Use np.nan for missing values
                    except:
                        lambda_dual[t, s] = np.nan  # Use np.nan for errors
                        
                print("Model.fixed() dual extraction successful!")
            else:
                print("Fixed model optimization failed. Setting dual to zero.")
                lambda_dual = np.zeros((T, S))

            print("\nInternal Settlement Prices (Dual Variables):")
            for t in range(T):
                for s in range(S):
                    print(f"λ_{t}(ξ_{s}) = {lambda_dual[t, s]:.4f}")
                
    else:
        print("No optimal solution found.")
        lambda_dual = {(t, s): 0.0 for t in range(T) for s in range(S)}
        x_sol = yp_sol = ym_sol = dp_sol = dm_sol = z_sol = zc_sol = zd_sol = None
        original_objval = None
    
    return (x_sol, ep_sol, em_sol, yp_sol, ym_sol, dp_sol, dm_sol, z_sol, zc_sol, zd_sol, original_objval, lambda_dual)

x_hol, ep_hol, em_hol, yp_hol, ym_hol, dp_hol, dm_hol, z_hol, zc_hol, zd_hol, OBJ_HOL, lambda_dual = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)

Set parameter MIPGap to value 0.001
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  0.001

Optimize a model with 19780 rows, 31420 columns and 60100 nonzeros
Model fingerprint: 0x3a8364f3
Variable types: 21820 continuous, 9600 integer (9600 binary)
Coefficient statistics:
  Matrix range     [9e-01, 1e+00]
  Objective range  [2e+00, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 8e+02]
Presolve removed 11736 rows and 21688 columns
Presolve time: 0.04s
Presolved: 8044 rows, 9732 columns, 31389 nonzeros
Variable types: 9732 continuous, 0 integer (0 binary)

Root relaxation: objective 1.791372e+06, 6791 iterations, 0.06 seconds (0.08 work units)

    Nodes    |    Current Node    |     Objective Bounds      |     Work
 Expl Unexpl |  Obj  Depth 

In [97]:
case_found = [False] * 9

for i, t, s in product(range(I), range(T), range(S)):
    # 제약조건 1: yp_hol과 ym_hol이 동시에 양수인 경우
    if not case_found[0] and yp_hol[i, t, s] > 0 and ym_hol[i, t, s] > 0:
        print(f"Case 1 발생: i={i}, t={t}, s={s} - yp_hol={yp_hol[i,t,s]}, ym_hol={ym_hol[i,t,s]}")
        case_found[0] = True
    
    # 제약조건 2: ym_hol과 zc_hol이 동시에 양수인 경우
    if not case_found[1] and ym_hol[i, t, s] > 0 and zc_hol[i, t, s] > 0:
        print(f"Case 2 발생: i={i}, t={t}, s={s} - ym_hol={ym_hol[i,t,s]}, zc_hol={zc_hol[i,t,s]}")
        case_found[1] = True
    
    # 제약조건 3: zc_hol과 zd_hol이 동시에 양수인 경우
    if not case_found[2] and zc_hol[i, t, s] > 0 and zd_hol[i, t, s] > 0:
        print(f"Case 3 발생: i={i}, t={t}, s={s} - zc_hol={zc_hol[i,t,s]}, zd_hol={zd_hol[i,t,s]}")
        case_found[2] = True
    
    # 제약조건 4: dp_hol과 dm_hol이 동시에 양수인 경우
    if not case_found[3] and dp_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 4 발생: i={i}, t={t}, s={s} - dp_hol={dp_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}")
        case_found[3] = True
    
    # 제약조건 5: zc_hol과 dm_hol이 동시에 양수인 경우
    if not case_found[4] and zc_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 5 발생: i={i}, t={t}, s={s} - zc_hol={zc_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}")
        case_found[4] = True
    
    # 제약조건 6: yp_hol과 dm_hol이 동시에 양수인 경우
    if not case_found[5] and yp_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 6 발생: i={i}, t={t}, s={s} - yp_hol={yp_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}")
        case_found[5] = True
    
    # 제약조건 7: ym_hol과 dp_hol이 동시에 양수인 경우
    if not case_found[6] and ym_hol[i, t, s] > 0 and dp_hol[i, t, s] > 0:
        print(f"Case 7 발생: i={i}, t={t}, s={s} - ym_hol={ym_hol[i,t,s]}, dp_hol={dp_hol[i,t,s]}")
        case_found[6] = True
    
    # 제약조건 8: ep_hol과 dm_hol이 동시에 양수인 경우
    if not case_found[7] and ep_hol[i, t, s] > 0 and dm_hol[i, t, s] > 0:
        print(f"Case 8 발생: i={i}, t={t}, s={s} - ep_hol={ep_hol[i,t,s]}, dm_hol={dm_hol[i,t,s]}")
        case_found[7] = True
    
    # 제약조건 9: em_hol과 dp_hol이 동시에 양수인 경우
    if not case_found[8] and em_hol[i, t, s] > 0 and dp_hol[i, t, s] > 0:
        print(f"Case 9 발생: i={i}, t={t}, s={s} - em_hol={em_hol[i,t,s]}, dp_hol={dp_hol[i,t,s]}")
        case_found[8] = True
    
    # 모든 케이스가 발견되면 조기 종료
    if all(case_found):
        break

# 발견되지 않은 케이스 출력
for i, found in enumerate(case_found, 1):
    if not found:
        print(f"Case {i}: 발생하지 않음")

Case 2 발생: i=0, t=16, s=0 - ym_hol=510.0, zc_hol=17.0
Case 5 발생: i=0, t=16, s=0 - zc_hol=17.0, dm_hol=510.0
Case 1 발생: i=0, t=16, s=11 - yp_hol=28.0, ym_hol=510.0
Case 4 발생: i=0, t=16, s=11 - dp_hol=28.0, dm_hol=510.0
Case 6 발생: i=0, t=16, s=11 - yp_hol=28.0, dm_hol=510.0
Case 7 발생: i=0, t=16, s=11 - ym_hol=510.0, dp_hol=28.0
Case 8 발생: i=0, t=16, s=11 - ep_hol=28.0, dm_hol=510.0
Case 9 발생: i=0, t=16, s=11 - em_hol=510.0, dp_hol=28.0
Case 3: 발생하지 않음


In [105]:
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    41.20     0.00     1.70     0.00     0.00     0.00    39.50     0.00    14.08
 9 |   184.70    41.48    22.56    16.85    16.85    16.85   137.51     0.00    50.42
10 |   440.05   231.44   156.34    90.42    90.42    90.42   145.45     2.76   176.93
11 |   655.15   197.00   434.62   145.18   145.18   145.18   168.72     0.00   307.84
12 |   885.20     0.00   853.97     0.00     0.00     0.00    31.23     0.00   463.05
13 |  1324.65     0.00  1725.09     0.00     0.00     0.00     0.04   400.48   491.78
14 |  1425.80   848.00   837.53   687.98   681.83   681.83   429.70     1.45    70.26
15 |   842.85     0.00  1279.76     0.00     0.00     0.00     0.00   436.91   464.06
16 |   763.95   510.00   547.30   497.75   497.75   497.75   208.35     3.95     4.16
17 |   865.45     0.00  1013.70     0.00     0.00

### MILP로 dual 추출

In [ ]:
def optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1):
    set = gp.Model("set_independent")
    set.setParam("MIPGap", 1e-6)

    x_hol = set.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp")
    ym_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym")
    dp_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp")
    dm_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm")
    
    z_hol = set.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, name="z")
    zc_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, name="zc")
    zd_hol = set.addVars(I, T, S, vtype=GRB.CONTINUOUS, name="zd")

    phi1_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi1")
    phi2_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi2") 
    phi3_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi3")
    phi4_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi4")
    phi5_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi5")
    phi6_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi6")
    phi7_hol = set.addVars(I, T, S, vtype=GRB.BINARY, name="phi7")

    set.update()

    obj_hol = (
        gp.quicksum(P_DA[t] * x_hol[i, t] for i in range(I) for t in range(T)) +
        gp.quicksum((1/S) * (
            P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]
        ) for i in range(I) for t in range(T) for s in range(S))
    )
    
    set.setObjective(obj_hol, GRB.MAXIMIZE)

    for i, t, s in product(range(I), range(T), range(S)):
        set.addConstr(R[i, t, s] - x_hol[i, t] == yp_hol[i, t, s] - ym_hol[i, t, s] + dp_hol[i, t, s] - dm_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
        set.addConstr(R[i, t, s] + zd_hol[i, t, s] >= yp_hol[i, t, s] + dp_hol[i, t, s] + zc_hol[i, t, s])
        
        set.addConstr(zd_hol[i, t, s] <= z_hol[i, t, s])
        set.addConstr(zc_hol[i, t, s] <= K[i] - z_hol[i, t, s])
        set.addConstr(z_hol[i, t, s] <= K[i])
        set.addConstr(z_hol[i, t + 1, s] == z_hol[i, t, s] + zc_hol[i, t, s] - zd_hol[i, t, s])
        
        set.addConstr(yp_hol[i, t, s] <= M1 * phi1_hol[i, t, s]) ; set.addConstr(ym_hol[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
        set.addConstr(ym_hol[i, t, s] <= M1 * phi2_hol[i, t, s]) ; set.addConstr(zc_hol[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
        set.addConstr(zc_hol[i, t, s] <= M1 * phi3_hol[i, t, s]) ; set.addConstr(zd_hol[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
        set.addConstr(dp_hol[i, t, s] <= M1 * phi4_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
        set.addConstr(zc_hol[i, t, s] <= M1 * phi5_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
        set.addConstr(yp_hol[i, t, s] <= M1 * phi6_hol[i, t, s]) ; set.addConstr(dm_hol[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))
        set.addConstr(ym_hol[i, t, s] <= M1 * phi7_hol[i, t, s]) ; set.addConstr(dp_hol[i, t, s] <= M1 * (1 - phi7_hol[i, t, s]))

    for i, s in product(range(I), range(S)):
        set.addConstr(z_hol[i, 0, s] == K0[i])

    # 중요: balance constraint를 변수에 저장해야 dual variable 추출 가능
    balance_constraints = {}
    for t, s in product(range(T), range(S)):
        balance_constraints[t, s] = set.addConstr(
            gp.quicksum(dp_hol[i, t, s] for i in range(I)) == 
            gp.quicksum(dm_hol[i, t, s] for i in range(I)),
            name=f"balance_{t}_{s}"
        )

    for i in range(I):
        set.addConstr(
            gp.quicksum(P_DA[t] * x_hol[i, t] for t in range(T)) +
            gp.quicksum((1/S) * (P_RT[t, s] * yp_hol[i, t, s] - P_PN[t, s] * ym_hol[i, t, s]) for t in range(T) for s in range(S)) >= OBJ_IND[i]
        )

    set.optimize()
    
    if set.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {set.objVal}")
        
        # 먼저 원래 솔루션 저장
        x_sol = np.array([[x_hol[i, t].X for t in range(T)] for i in range(I)])
        yp_sol = np.array([[[yp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        ym_sol = np.array([[[ym_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        dp_sol = np.array([[[dp_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        dm_sol = np.array([[[dm_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        z_sol = np.array([[[z_hol[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
        zc_sol = np.array([[[zc_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        zd_sol = np.array([[[zd_hol[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
        original_objval = set.objVal
        
        # Dual variable 추출 시도
        try:
            # Method 1: 직접 dual variable 추출 시도
            lambda_dual = {}
            for t, s in product(range(T), range(S)):
                lambda_dual[t, s] = balance_constraints[t, s].Pi
            print("Direct dual extraction successful!")
            
        except AttributeError:
            print("Direct dual extraction failed. Using Model.fixed() method...")
            
            # Method 2: Model.fixed()를 사용하여 이진 변수들을 현재값으로 고정
            fixed_model = set.fixed()
            fixed_model.setParam("OutputFlag", 0)  # 출력 억제
            
            # 고정된 모델 최적화
            fixed_model.optimize()
            
            # 목적함수값 비교 확인
            if fixed_model.status == GRB.OPTIMAL:
                original_obj = set.objVal
                fixed_obj = fixed_model.objVal
                obj_diff = abs(original_obj - fixed_obj)
                
                print(f"Original MIP objective: {original_obj:.6f}")
                print(f"Fixed LP objective: {fixed_obj:.6f}")
                print(f"Difference: {obj_diff:.10f}")
                
                if obj_diff < 1e-6:
                    print("✅ Objective values match! Fixed model is correct.")
                else:
                    print("⚠️  Warning: Objective values don't match. Check model consistency.")
            
            # Fixed model에서 balance constraint의 dual variable 추출
            lambda_dual = {}
            if fixed_model.status == GRB.OPTIMAL:
                # Balance constraint의 이름으로 찾아서 dual variable 추출
                for t, s in product(range(T), range(S)):
                    constr_name = f"balance_{t}_{s}"
                    try:
                        # Constraint를 이름으로 찾기
                        constr = fixed_model.getConstrByName(constr_name)
                        if constr is not None:
                            lambda_dual[t, s] = constr.Pi
                        else:
                            lambda_dual[t, s] = 0.0
                    except:
                        lambda_dual[t, s] = 0.0
                        
                print("Model.fixed() dual extraction successful!")
            else:
                print("Fixed model optimization failed. Setting dual to zero.")
                lambda_dual = {(t, s): 0.0 for t in range(T) for s in range(S)}
        
        print("\nInternal Settlement Prices (Dual Variables):")
        for t in range(T):
            for s in range(S):
                print(f"λ_{t}(ξ_{s}) = {lambda_dual[t, s]:.4f}")
                
    else:
        print("No optimal solution found.")
        lambda_dual = {(t, s): 0.0 for t in range(T) for s in range(S)}
        x_sol = yp_sol = ym_sol = dp_sol = dm_sol = z_sol = zc_sol = zd_sol = None
        original_objval = None
    
    return (x_sol, yp_sol, ym_sol, dp_sol, dm_sol, z_sol, zc_sol, zd_sol, original_objval, lambda_dual)

x_hol, yp_hol, ym_hol, dp_hol, dm_hol, z_hol, zc_hol, zd_hol, OBJ_HOL, lambda_dual = optimize_hol(R, K, K0, P_DA, P_RT, P_PN, I, T, S, M1)

Set parameter MIPGap to value 1e-06
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
MIPGap  1e-06

Optimize a model with 121455 rows, 84370 columns and 312370 nonzeros
Model fingerprint: 0xf1f9eee5
Variable types: 42370 continuous, 42000 integer (42000 binary)
Coefficient statistics:
  Matrix range     [9e-01, 8e+02]
  Objective range  [9e-01, 2e+02]
  Bounds range     [1e+00, 1e+00]
  RHS range        [1e+00, 4e+05]
Presolve removed 47255 rows and 28400 columns
Presolve time: 0.48s
Presolved: 74200 rows, 55970 columns, 200921 nonzeros
Variable types: 28651 continuous, 27319 integer (27319 binary)
Deterministic concurrent LP optimizer: primal and dual simplex
Showing primal log only...

Concurrent spin time: 0.10s

Solved with dual simplex

Root relaxation: objective 1.

In [ ]:
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    46.37     0.00    13.13     0.00     0.00     0.00    37.37     4.13    15.67
 9 |   169.57    23.00    11.87     0.00     0.13     0.13   135.37     0.67    48.90
10 |   438.10   206.00    83.40     0.00    24.87    24.87   174.33    25.63   183.60
11 |   661.03     0.00   588.33     0.00     0.00     0.00   137.07    64.37   332.30
12 |   875.37     0.00   800.07     0.00     0.00     0.00    90.87    15.57   405.00
13 |  1259.70     0.00  1645.62     0.00     0.00     0.00     2.23   388.15   480.30
14 |  1365.03   835.00   214.30    54.68   518.69   518.69   370.42     0.00    94.38
15 |   868.27     0.00  1299.13     0.00     0.00     0.00     6.30   437.17   464.80
16 |   800.70   363.00   200.97     0.00   310.87   310.87   238.13     1.40    33.93
17 |   763.67     0.00   982.60     0.00     0.00

### Individual Replay

In [175]:
import gurobipy as gp
from gurobipy import GRB
from itertools import product
import numpy as np

def individual_replay(R, K, K0, P_DA, P_RT, P_PN, lambda_dual, I, T, S, M1,
                      phi1_hol, phi2_hol, phi3_hol, phi4_hol, phi5_hol, phi6_hol, phi7_hol):
    
    model = gp.Model("DER_Individual_Replay")
    # model.setParam("Heuristics", 0.2)
    model.setParam("TimeLimit", 60*15)
    
    x = model.addVars(I, T, vtype=GRB.CONTINUOUS, lb=0, name="x")
    yp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="yp") 
    ym = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="ym") 
    dp = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dp") 
    dm = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="dm") 
    z = model.addVars(I, T + 1, S, vtype=GRB.CONTINUOUS, lb=0, name="z")
    zc = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zc") 
    zd = model.addVars(I, T, S, vtype=GRB.CONTINUOUS, lb=0, name="zd")
    
    model.update()

    # obj = (
    #     gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
    #     gp.quicksum((1/S) * (
    #         P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s] + lambda_dual[t, s] * dp[i, t, s] - lambda_dual[t, s] * dm[i, t, s]
    #     ) for i in range(I) for t in range(T) for s in range(S))
    # )

    # obj = (
    #     gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) +
    #     gp.quicksum((1/S) * (
    #         P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
    #     ) for i in range(I) for t in range(T) for s in range(S)) +
    #     gp.quicksum((1/S) * lambda_dual[t, s] * (
    #         gp.quicksum(dp[i, t, s] for i in range(I)) - 
    #         gp.quicksum(dm[i, t, s] for i in range(I))
    #     ) for t in range(T) for s in range(S))
    # )

    obj = (
        gp.quicksum(P_DA[t] * x[i, t] for i in range(I) for t in range(T)) + 
        gp.quicksum((1/S) * (
            P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]
        ) for i in range(I) for t in range(T) for s in range(S)) +
        gp.quicksum(lambda_dual[t, s] * (
            gp.quicksum(dm[i, t, s] for i in range(I)) - 
            gp.quicksum(dp[i, t, s] for i in range(I))
        ) for t in range(T) for s in range(S))
    )
    
    model.setObjective(obj, GRB.MAXIMIZE)
    
    for i, t, s in product(range(I), range(T), range(S)):
        model.addConstr(R[i, t, s] - x[i, t] == yp[i, t, s] - ym[i, t, s] + dp[i, t, s] - dm[i, t, s] + zc[i, t, s] - zd[i, t, s])
        model.addConstr(R[i, t, s] + zd[i, t, s] >= yp[i, t, s] + dp[i, t, s] + zc[i, t, s])
        
        model.addConstr(zd[i, t, s] <= z[i, t, s])
        model.addConstr(zc[i, t, s] <= K[i] - z[i, t, s])
        model.addConstr(z[i, t, s] <= K[i])
        model.addConstr(z[i, t + 1, s] == z[i, t, s] + 0.92 * zc[i, t, s] - zd[i, t, s] / 0.95)
        
        model.addConstr(yp[i, t, s] <= M1 * phi1_hol[i, t, s]) ; model.addConstr(ym[i, t, s] <= M1 * (1 - phi1_hol[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi2_hol[i, t, s]) ; model.addConstr(zc[i, t, s] <= M1 * (1 - phi2_hol[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi3_hol[i, t, s]) ; model.addConstr(zd[i, t, s] <= M1 * (1 - phi3_hol[i, t, s]))
        model.addConstr(dp[i, t, s] <= M1 * phi4_hol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi4_hol[i, t, s]))
        model.addConstr(zc[i, t, s] <= M1 * phi5_hol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi5_hol[i, t, s]))
        model.addConstr(yp[i, t, s] <= M1 * phi6_hol[i, t, s]) ; model.addConstr(dm[i, t, s] <= M1 * (1 - phi6_hol[i, t, s]))
        model.addConstr(ym[i, t, s] <= M1 * phi7_hol[i, t, s]) ; model.addConstr(dp[i, t, s] <= M1 * (1 - phi7_hol[i, t, s]))

    for i, s in product(range(I), range(S)):
        model.addConstr(z[i, 0, s] == K0[i])
    
    for i in range(I):
        model.addConstr(
            gp.quicksum(P_DA[t] * x[i, t] for t in range(T)) +
            gp.quicksum((1/S) * (P_RT[t, s] * yp[i, t, s] - P_PN[t, s] * ym[i, t, s]) for t in range(T) for s in range(S)) >= OBJ_IND[i]
        )

    model.optimize()
    
    if model.status == GRB.OPTIMAL:
        print(f"Optimal solution found! Objective value: {model.objVal}")
    else:
        print("No optimal solution found.")
    
    x_sol = np.array([[x[i, t].X for t in range(T)] for i in range(I)])
    yp_sol = np.array([[[yp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    ym_sol = np.array([[[ym[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dp_sol = np.array([[[dp[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    dm_sol = np.array([[[dm[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    z_sol = np.array([[[z[i, t, s].X for s in range(S)] for t in range(T+1)] for i in range(I)])
    zc_sol = np.array([[[zc[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    zd_sol = np.array([[[zd[i, t, s].X for s in range(S)] for t in range(T)] for i in range(I)])
    
    return (x_sol, yp_sol, ym_sol, z_sol, zc_sol, zd_sol, dp_sol, dm_sol, model.objVal)

In [176]:
x_re, yp_re, ym_re, z_re, zc_re, zd_re, dp_re, dm_re, OBJ_RE = individual_replay(R, K, K0, P_DA, P_RT, P_PN, lambda_dual, I, T, S, M1, phi1_hol, phi2_hol, phi3_hol, phi4_hol, phi5_hol, phi6_hol, phi7_hol)

Set parameter TimeLimit to value 900
Gurobi Optimizer version 12.0.2 build v12.0.2rc0 (win64 - Windows 11.0 (26100.2))

CPU model: 13th Gen Intel(R) Core(TM) i9-13900K, instruction set [SSE2|AVX|AVX2]
Thread count: 24 physical cores, 32 logical processors, using up to 32 threads

Non-default parameters:
TimeLimit  900

Optimize a model with 36080 rows, 12795 columns and 64995 nonzeros
Model fingerprint: 0xa420d7ba
Coefficient statistics:
  Matrix range     [9e-01, 2e+02]
  Objective range  [3e+00, 2e+02]
  Bounds range     [0e+00, 0e+00]
  RHS range        [1e+00, 5e+05]
Presolve removed 33175 rows and 9407 columns
Presolve time: 0.03s
Presolved: 2905 rows, 3388 columns, 10200 nonzeros

Concurrent LP optimizer: primal simplex, dual simplex, and barrier
Showing barrier log only...

Ordering time: 0.00s

Barrier statistics:
 AA' NZ     : 1.850e+04
 Factor NZ  : 5.934e+04 (roughly 3 MB of memory)
 Factor Ops : 1.905e+06 (less than 1 second per iteration)
 Threads    : 1

                 

In [178]:
# 시나리오 평균으로 출력
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x_re[:, t].sum()
    yp_avg = np.mean([yp_re[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym_re[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_re[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm_re[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_re[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_re[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_re[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    41.07     0.00     4.49     0.00     0.00     0.00    37.33     0.76    13.98
 9 |   190.27  6961.00     0.00     0.00    64.69  6958.27   122.84     0.00    47.53
10 |   420.07   209.00    78.46     0.00    53.46    42.02   124.00     2.82   160.55
11 |   598.13   295.76   145.76     0.00    69.97    44.38   148.30    17.27   271.65
12 |   853.33     0.00   751.03     0.00     0.00     0.00   102.30     0.00   389.91
13 |  1300.27     0.00  1580.50     0.00     0.00     0.00     1.21   281.44   484.02
14 |  1405.60  7362.03    94.82     0.00   531.27  6837.68   261.28     6.12   188.88
15 |   825.67     0.00  1199.07     0.00     0.00     0.00     0.00   373.40   422.82
16 |   759.27  7325.00    70.80     0.00   216.80  6988.94   163.88    28.28    29.77
17 |   820.33     0.00   963.57     0.00     0.00

In [170]:
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print("HOLISTIC")
print(header)

for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[:, t, s].sum() for s in range(S)])
    x_sum = x_hol[:, t].sum()
    yp_avg = np.mean([yp_hol[:, t, s].sum() for s in range(S)])
    ym_avg = np.mean([ym_hol[:, t, s].sum() for s in range(S)])
    dp_avg = np.mean([dp_hol[:, t, s].sum() for s in range(S)])
    dm_avg = np.mean([dm_hol[:, t, s].sum() for s in range(S)])
    zc_avg = np.mean([zc_hol[:, t, s].sum() for s in range(S)])
    zd_avg = np.mean([zd_hol[:, t, s].sum() for s in range(S)])
    z_avg = np.mean([z_hol[:, t, s].sum() for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

HOLISTIC
 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    41.07     0.00     4.22     0.00     0.00     0.00    37.60     0.76    15.09
 9 |   190.27    55.99     0.40     0.00    53.25    53.25   133.88     0.00    48.89
10 |   420.07   232.87    64.28     0.00    62.61    62.61   124.17     1.25   172.05
11 |   598.13   317.00   142.74     0.00    59.66    59.66   153.17    14.78   284.97
12 |   853.33     0.00   768.39     0.00     0.00     0.00    84.95     0.00   410.33
13 |  1300.27     0.00  1598.87     0.00     0.00     0.00     1.21   299.81   488.48
14 |  1405.60   919.00   215.76     0.00   429.80   429.80   276.11     5.27   174.01
15 |   825.67     0.00  1217.29     0.00     0.00     0.00     0.00   391.62   422.49
16 |   759.27   524.00    86.54     0.00   209.36   209.36   157.66     8.93    10.25
17 |   820.33     0.00   955.14     0.00

In [36]:
# 한 명 시나리오 평균으로 출력
header = (
    f"{'t':>2} | "
    f"{'R':>8} {'x':>8} {'y+':>8} {'y-':>8} "
    f"{'d+':>8} {'d-':>8} {'zc':>8} {'zd':>8} {'z':>8}\n"
    + "-" * 90
)
print(header)
i=0
for t in range(8, 22):
    # 각 변수의 시나리오 평균 계산
    R_avg = np.mean([R[i, t, s] for s in range(S)])
    x_sum = x_re[i, t]
    yp_avg = np.mean([yp_re[i, t, s] for s in range(S)])
    ym_avg = np.mean([ym_re[i, t, s] for s in range(S)])
    dp_avg = np.mean([dp_re[i, t, s] for s in range(S)])
    dm_avg = np.mean([dm_re[i, t, s] for s in range(S)])
    zc_avg = np.mean([zc_re[i, t, s] for s in range(S)])
    zd_avg = np.mean([zd_re[i, t, s] for s in range(S)])
    z_avg = np.mean([z_re[i, t, s] for s in range(S)])
    
    print(
        f"{t:>2} | "
        f"{R_avg:>8.2f} {x_sum:>8.2f} {yp_avg:>8.2f} {ym_avg:>8.2f} "
        f"{dp_avg:>8.2f} {dm_avg:>8.2f} {zc_avg:>8.2f} {zd_avg:>8.2f} {z_avg:>8.2f}"
    )

 t |        R        x       y+       y-       d+       d-       zc       zd        z
------------------------------------------------------------------------------------------
 8 |    16.94     0.00     0.76     0.00     0.00     0.00    16.18     0.00     0.00
 9 |     3.60     1.00     0.28     0.00     0.00     0.00     2.32     0.00    16.18
10 |    35.04     0.00     2.56     0.00     0.00     0.00    32.48     0.00    18.50
11 |    60.76     0.00    27.66     0.00     0.00     0.00    33.28     0.18    50.98
12 |   136.64     0.00   122.42     0.00     0.00     0.00    15.06     0.84    84.08
13 |   208.24     0.00   264.10     0.00     0.00     0.00     0.84    56.70    98.30
14 |   209.64   182.00    16.50    10.00     0.00    12.22    35.26     1.90    42.44
15 |   174.18     0.00   245.98     0.00     0.00     0.00     0.00    71.80    75.80
16 |    19.12    33.00     0.02     0.00     0.00    13.90     0.00     0.00     4.00
17 |     0.00     0.00     4.00     0.00     0.00